# Unemployment rate vs obligations of the department in the region
- Comparison of  Department of Labor and Employment (DOLE) obligations against regional unemployment rates from 2021–2025.

In [0]:
%sql

WITH dole_obligations AS (

    -- Aggregate DOLE obligations per region and year
    SELECT
        year_id,
        location_id,
        SUM(obligation_amount) AS dole_obligations

    FROM pondolens.dev_gold.fact_regional_expenditure

    WHERE particular_id = 30021

    GROUP BY
        year_id,
        location_id
),

regional_unemployment AS (

    -- Retrieve unemployment rate per region and year
    SELECT
        f.year_id,
        f.location_id,
        f.measurement AS unemployment_rate

    FROM pondolens.dev_gold.fact_regional_indicator f

    INNER JOIN pondolens.dev_gold.dim_indicator i
        ON f.indicator_id = i.indicator_id

    WHERE LOWER(TRIM(i.indicator)) = 'unemployment'
      AND LOWER(TRIM(i.measure_type)) = 'unemployment_rate'
),

combined_data AS (

    SELECT
        o.year_id,
        o.location_id,
        o.dole_obligations,
        u.unemployment_rate

    FROM dole_obligations o

    INNER JOIN regional_unemployment u
        ON o.year_id = u.year_id
       AND o.location_id = u.location_id
)

SELECT
    y.year,    r.region_code,
    r.region,

    ROUND(c.dole_obligations, 2) AS dole_obligations,
    ROUND(c.unemployment_rate, 2) AS unemployment_rate

FROM combined_data c

INNER JOIN pondolens.dev_gold.dim_year y
    ON c.year_id = y.year_id

INNER JOIN pondolens.dev_gold.dim_region r
    ON c.location_id = r.region_id

WHERE LOWER(TRIM(r.geographic_level)) = 'region'
  AND y.year BETWEEN 2021 AND 2025

ORDER BY
    r.region,
    y.year;
